In [1]:
import sys
sys.path.append('../')
import os
os.environ['PATH'] = os.environ['PATH'] + os.pathsep + '/home/ubuntu/miniconda3/envs/utrtool/bin/'

import pybedtools

from utrcalling import utr_reference as utr_ref
from utrcalling import platforms

In [2]:
gtf_path = "../tests/test_data/test_mus_musculus_GRCm38_102.gtf"
%time utr = utr_ref.extract_feature(gtf_path, 'three_prime_utr')
print(utr)

2021-06-30 14:06:14,676 - INFO - Parsing GTF reference file to retrieve only the UTR annotations
INFO:root:Extracted GTF attributes: ['gene_id', 'gene_version', 'transcript_id', 'transcript_version', 'exon_number', 'gene_name', 'gene_source', 'gene_biotype', 'havana_gene', 'havana_gene_version', 'transcript_name', 'transcript_source', 'transcript_biotype', 'protein_id', 'protein_version', 'tag', 'transcript_support_level', 'ccds_id', 'exon_id', 'exon_version', 'havana_transcript', 'havana_transcript_version']
2021-06-30 14:06:14,767 - INFO - Finished parsing GTF file
CPU times: user 93.5 ms, sys: 0 ns, total: 93.5 ms
Wall time: 92.4 ms
     seqname          source          feature      start        end  score  \
4          1         ensembl  three_prime_utr  130697017  130697884    NaN   
5          1         ensembl  three_prime_utr  170885460  170885881    NaN   
6          1         ensembl  three_prime_utr   77367185   77369671    NaN   
21         1  ensembl_havana  three_prime_ut

In [3]:
feature_list = utr["feature"].unique()
print(feature_list)

['three_prime_utr']


In [4]:
utr['seqname'].unique()

array(['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19',
       '2', '3', '4', '5', '6', '7', '8', '9', 'X', 'Y'], dtype=object)

In [5]:
utr = utr.sort_values(by=['seqname', 'start'])  # sort by chr and start for bed merge
utr.fillna(value={'score': 0}, inplace=True)
print(utr)

     seqname          source          feature      start        end  score  \
100        1          havana  three_prime_utr    7160892    7161063    0.0   
107        1          havana  three_prime_utr    9552943    9552976    0.0   
108        1          havana  three_prime_utr    9553318    9553472    0.0   
21         1  ensembl_havana  three_prime_utr   10024601   10024978    0.0   
70         1          havana  three_prime_utr   15843083   15843470    0.0   
...      ...             ...              ...        ...        ...    ...   
1980       X          havana  three_prime_utr  166189501  166189820    0.0   
1981       X          havana  three_prime_utr  167476787  167478076    0.0   
1982       X          havana  three_prime_utr  167497813  167497949    0.0   
1950       X  ensembl_havana  three_prime_utr  169776378  169776476    0.0   
1999       Y          havana  three_prime_utr    1224497    1224546    0.0   

     strand frame             gene_id gene_version  ...  \
100 

In [6]:
annotations = ['gene_id', 'score', 'strand', 'transcript_id', 'gene_name']
columns, ann_idx = utr_ref.prepare_annotations_index(annotations)
utr_bed = pybedtools.BedTool.from_dataframe(utr, columns=columns)

utr_reduced = utr_ref.reduce_regions(utr_bed,
annotations_index=ann_idx)  # find and merge overlapping regions


2021-06-30 14:06:14,895 - INFO - Merging overlapping UTR regions


In [7]:
print(utr)

     seqname          source          feature      start        end  score  \
100        1          havana  three_prime_utr    7160892    7161063    0.0   
107        1          havana  three_prime_utr    9552943    9552976    0.0   
108        1          havana  three_prime_utr    9553318    9553472    0.0   
21         1  ensembl_havana  three_prime_utr   10024601   10024978    0.0   
70         1          havana  three_prime_utr   15843083   15843470    0.0   
...      ...             ...              ...        ...        ...    ...   
1980       X          havana  three_prime_utr  166189501  166189820    0.0   
1981       X          havana  three_prime_utr  167476787  167478076    0.0   
1982       X          havana  three_prime_utr  167497813  167497949    0.0   
1950       X  ensembl_havana  three_prime_utr  169776378  169776476    0.0   
1999       Y          havana  three_prime_utr    1224497    1224546    0.0   

     strand frame             gene_id gene_version  ...  \
100 

In [8]:
print(utr_reduced.head())

1	7160892	7161063	ENSMUSG00000051285	0.0	+	ENSMUST00000182675	Pcmtd1
 1	9552943	9552976	ENSMUSG00000025911	0.0	+	ENSMUST00000130927	Adhfe1
 1	9553318	9553472	ENSMUSG00000025911	0.0	+	ENSMUST00000130927	Adhfe1
 1	10024601	10024978	ENSMUSG00000025917	0.0	-	ENSMUST00000027050	Cops5
 1	15843083	15843470	ENSMUSG00000025925	0.0	+	ENSMUST00000027057	Terf1
 1	30944938	30945030	ENSMUSG00000117310	0.0	-	ENSMUST00000233506	Ptp4a1
 1	33734702	33734858	ENSMUSG00000004768	0.0	+	ENSMUST00000138024	Rab23
 1	34552334	34552678	ENSMUSG00000037529	0.0	-	ENSMUST00000190790	Prss40
 1	34811830	34812739	ENSMUSG00000037509	0.0	+	ENSMUST00000159747	Arhgef4
 1	36428102	36428216	ENSMUSG00000001143	0.0	-	ENSMUST00000001171	Lman2l
 None


In [9]:
utr_reduced = utr_ref.generate_utr_name(utr_reduced)
utr_reduced.head()

,chrom,start,end,name,score,strand,thickStart,thickEnd
utr_name,,,,,,,,
ENSMUSG00000051285,1,7160892,7161063,ENSMUSG00000051285,0.0,+,ENSMUST00000182675,Pcmtd1
ENSMUSG00000025911,1,9552943,9552976,ENSMUSG00000025911,0.0,+,ENSMUST00000130927,Adhfe1
ENSMUSG00000025911_2,1,9553318,9553472,ENSMUSG00000025911,0.0,+,ENSMUST00000130927,Adhfe1
ENSMUSG00000025917,1,10024601,10024978,ENSMUSG00000025917,0.0,-,ENSMUST00000027050,Cops5
ENSMUSG00000025925,1,15843083,15843470,ENSMUSG00000025925,0.0,+,ENSMUST00000027057,Terf1


In [10]:
len(utr_reduced.index.values)

1016

In [11]:
utr_reduced = utr_ref.correct_column_names(utr_reduced)
utr_reduced.head()

,chrom,start,end,name,score,strand,transcript_id,gene_name
utr_name,,,,,,,,
ENSMUSG00000051285,1,7160892,7161063,ENSMUSG00000051285,0.0,+,ENSMUST00000182675,Pcmtd1
ENSMUSG00000025911,1,9552943,9552976,ENSMUSG00000025911,0.0,+,ENSMUST00000130927,Adhfe1
ENSMUSG00000025911_2,1,9553318,9553472,ENSMUSG00000025911,0.0,+,ENSMUST00000130927,Adhfe1
ENSMUSG00000025917,1,10024601,10024978,ENSMUSG00000025917,0.0,-,ENSMUST00000027050,Cops5
ENSMUSG00000025925,1,15843083,15843470,ENSMUSG00000025925,0.0,+,ENSMUST00000027057,Terf1


In [12]:
from utrcalling import utr_mapping

# check if aligments are correctly parsed: (each should output True)
print(all(alignment.proper_pair for alignment in alignments))
print(not any(alignment.pcr_or_optical_duplicate for alignment in alignments))
print(not any(alignment.not_primary_alignment for alignment in alignments))

In [13]:
intervals = utr_mapping.generate_utr_intervals(utr_reduced)
ct = 0
for name, interval in intervals.items(): 
    print(name, interval, interval.start_d, sep=': ')
    ct += 1
    if ct == 5:
        break

Generating genomic interval for UTR number: 1016it [00:00, 396964.40it/s]ENSMUSG00000051285: 1:[7160891,7161063)/+: 7160891
ENSMUSG00000025911: 1:[9552942,9552976)/+: 9552942
ENSMUSG00000025911_2: 1:[9553317,9553472)/+: 9553317
ENSMUSG00000025917: 1:[10024600,10024978)/-: 10024977
ENSMUSG00000025925: 1:[15843082,15843470)/+: 15843082



In [14]:
print(intervals['ENSMUSG00000005804'], intervals['ENSMUSG00000025917'], sep='\n')

2:[122746630,122749475)/+
1:[10024600,10024978)/-


In [15]:
alignments = ["../tests/test_data/test_UTRoldNSCgenome_bwt2_102_sort.bam", "../tests/test_data/test_UTRyoungNSCgenome_bwt2_102_sort.bam"]
platform = platforms.Platform(platform='10x', version=2)
read_mappings = [utr_mapping.map_reads_to_utr(alignment, intervals, platform, read='first') for alignment in alignments]

2021-06-30 14:06:15,256 - INFO - Extracting reads from ../tests/test_data/test_UTRoldNSCgenome_bwt2_102_sort.bam and mapping them to the reference UTR
Mapping reads to reference UTR number: 100%|██████████| 1009/1009 [00:00<00:00, 5009.41it/s]
2021-06-30 14:06:15,474 - DEBUG - Size of mapped reads object in bytes: 351752        

Memory usage:13.2
2021-06-30 14:06:15,474 - INFO - Extracting reads from ../tests/test_data/test_UTRyoungNSCgenome_bwt2_102_sort.bam and mapping them to the reference UTR
Mapping reads to reference UTR number: 100%|██████████| 1009/1009 [00:00<00:00, 4818.61it/s]
2021-06-30 14:06:15,700 - DEBUG - Size of mapped reads object in bytes: 385872        

Memory usage:13.2


In [16]:
print(read_mappings[0].ENSMUSG00000005804)

MappedReadInformation(utr_name='ENSMUSG00000005804', read_start_positions=[122748098, 122748088, 122748096], read_strands=['-', '-', '-'], read_barcodes=['GATGAAACATTAACCG', 'TGCGCAGAGTCCATAC', 'GCTGGGTGTACGCACC'], read_umis=['CAACACAAG', 'ATCGAGAGT', 'CTGAGTATG'], read_lengths=[51, 37, 44], utr_sizes_from_reads=[1468, 1458, 1466])


In [17]:
dct = read_mappings[0].todict()
for key, value in dct.items():
    print(key, value, sep='\n')
    break


ENSMUSG00000025917
{'utr_name': 'ENSMUSG00000025917', 'read_start_positions': [10024824, 10024829, 10024829, 10024833], 'read_strands': ['+', '+', '+', '+'], 'read_barcodes': ['CGTTGGGCAGTCAGAG', 'CCGTGGAGTCTCGTTC', 'GTAGTCAAGCTATGCT', 'TGCGCAGAGGCATGTG'], 'read_umis': ['TGGCTTATA', 'AACGTAACA', 'ACCTCCGGT', 'ACCCGCACT'], 'read_lengths': [46, 50, 48, 46], 'utr_sizes_from_reads': [153, 148, 148, 144]}


In [18]:
read_mappings[1]['ENSMUSG00000005804', 'ENSMUSG00000025917']

[MappedReadInformation(utr_name='ENSMUSG00000005804', read_start_positions=[122748095, 122748095], read_strands=['-', '-'], read_barcodes=['AGACGTTAGTTGCAGG', 'GGCGACTGTAGCAAAT'], read_umis=['CTCATGGTA', 'ACCGGGTCC'], read_lengths=[44, 43], utr_sizes_from_reads=[1465, 1465]),
 MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024602, 10024826, 10024829, 10024829, 10024835, 10024839], read_strands=['+', '+', '+', '+', '+', '+'], read_barcodes=['CGACCTTTCTGCTGCT', 'TGTTCCGAGGGTCGAT', 'GTACGTAAGCGTTTAC', 'GTGCGGTAGGACAGCT', 'GCTGCAGCACTGTGTA', 'GTCCTCAGTAAACGCG'], read_umis=['TCACGTGAG', 'TCCTAAATG', 'ATGATCGGT', 'TCCAGGAAT', 'TTTAGAACT', 'GCTAGGACG'], read_lengths=[43, 45, 54, 50, 41, 41], utr_sizes_from_reads=[375, 151, 148, 148, 142, 138])]

In [19]:
read_mappings[0][0:4:2]

['ENSMUSG00000025917', 'ENSMUSG00000037509']

In [20]:
from utrcalling import peak_calling

/home/ubuntu/miniconda3/envs/utrtool/lib/python3.9/site-packages/anndata/core/anndata.py:17: FutureWarning: pandas.core.index is deprecated and will be removed in a future version.  The public classes are available in the top-level namespace.
  from pandas.core.index import RangeIndex


In [21]:
positions_counts_merged = peak_calling.merge_samples_utr_sizes(read_mappings)
print(len(positions_counts_merged), positions_counts_merged.ENSMUSG00000025917.todict(), sep='\n\n')

2021-06-30 14:06:16,632 - INFO - Merging all samples' UTR sizes
100%|██████████| 2/2 [00:00<00:00, 76.70it/s]71

{'utr_name': 'ENSMUSG00000025917', 'read_start_positions': [10024824, 10024829, 10024829, 10024833, 10024602, 10024826, 10024829, 10024829, 10024835, 10024839], 'read_strands': ['+', '+', '+', '+', '+', '+', '+', '+', '+', '+'], 'read_barcodes': ['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2'], 'read_umis': ['TGGCTTATA', 'AACGTAACA', 'ACCTCCGGT', 'ACCCGCACT', 'TCACGTGAG', 'TCCTAAATG', 'ATGATCGGT', 'TCCAGGAAT', 'TTTAGAACT', 'GCTAGGACG'], 'read_lengths': [46, 50, 48, 46, 43, 45, 54, 50, 41, 41], 'utr_sizes_from_reads': [153, 148, 148, 144, 375, 151, 148, 148, 142, 138]}



In [22]:
utr_subsetted = peak_calling.subset_reference_data(positions_counts_merged, utr_reduced)
print(f'sizes: subset: {len(utr_subsetted)}, original: {len(utr_reduced)}')


2021-06-30 14:06:16,748 - INFO - Subsetting UTR reference data
2021-06-30 14:06:16,749 - DEBUG - 71
2021-06-30 14:06:16,750 - DEBUG - 71
2021-06-30 14:06:16,750 - DEBUG - ______________________________________________________
2021-06-30 14:06:16,750 - DEBUG - 1016
2021-06-30 14:06:16,751 - DEBUG - False
2021-06-30 14:06:16,752 - DEBUG - 14
2021-06-30 14:06:16,753 - DEBUG -                      chrom      start        end                name  score  \
utr_name                                                                      
ENSMUSG00000021952      14   57603586   57603731  ENSMUSG00000021952    0.0   
ENSMUSG00000021952      14   57643335   57643451  ENSMUSG00000021952    0.0   
ENSMUSG00000034206      16   37029343   37029490  ENSMUSG00000034206    0.0   
ENSMUSG00000034206      16   37094501   37094992  ENSMUSG00000034206    0.0   
ENSMUSG00000041119      17   31466355   31466468  ENSMUSG00000041119    0.0   
ENSMUSG00000041119_2    17   31469794   31469898  ENSMUSG00000041119   

ValueError: cannot reindex from a duplicate axis

In [23]:
utr_subsetted.iloc[0]

chrom                             1
start                      10024601
end                        10024978
name             ENSMUSG00000025917
score                           0.0
strand                            -
transcript_id    ENSMUST00000027050
gene_name                     Cops5
Name: ENSMUSG00000025917, dtype: object

In [24]:
utr_subsetted.head()

,chrom,start,end,name,score,strand,transcript_id,gene_name
utr_name,,,,,,,,
ENSMUSG00000025917,1,10024601,10024978,ENSMUSG00000025917,0.0,-,ENSMUST00000027050,Cops5
ENSMUSG00000025925,1,15843083,15843470,ENSMUSG00000025925,0.0,+,ENSMUST00000027057,Terf1
ENSMUSG00000037509,1,34811830,34812739,ENSMUSG00000037509,0.0,+,ENSMUST00000159747,Arhgef4
ENSMUSG00000026356,1,128412118,128412347,ENSMUSG00000026356,0.0,-,ENSMUST00000064309,Dars
ENSMUSG00000058267,1,160199611,160199812,ENSMUSG00000058267,0.0,+,ENSMUST00000097193,Mrps14


In [25]:
%time ft = peak_calling.filter_utr_sizes(positions_counts_merged)

2021-06-30 11:25:23,990 - INFO - Filtering samples UTR sizes
CPU times: user 10.3 ms, sys: 4.62 ms, total: 14.9 ms
Wall time: 14.5 ms


In [26]:
import pandas as pd
vivo_anno_path = "../tests/test_data/vivo_anno.csv"
vivo_anno = pd.read_csv(vivo_anno_path)

In [27]:
vivo_anno.head()

,CellBarcode,CellType,CellAge,pTime
0,AAACCTGCACCAGTTA-1,qNSC1,old,4.018431
1,AAACCTGCATGGTCAT-1,qNSC2,old,14.861303
2,AAACGGGCACCGGAAA-1,qNSC2,old,20.304435
3,AAACGGGCAGCAGTTT-1,qNSC2,old,10.686136
4,AAACGGGGTAAGTTCC-1,qNSC2,old,10.351203


In [28]:
len(ft)

71

In [29]:
ft.ENSMUSG00000025917

MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024824, 10024829, 10024829, 10024833, 10024602, 10024826, 10024829, 10024829, 10024835, 10024839], read_strands=['+', '+', '+', '+', '+', '+', '+', '+', '+', '+'], read_barcodes=['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2'], read_umis=['TGGCTTATA', 'AACGTAACA', 'ACCTCCGGT', 'ACCCGCACT', 'TCACGTGAG', 'TCCTAAATG', 'ATGATCGGT', 'TCCAGGAAT', 'TTTAGAACT', 'GCTAGGACG'], read_lengths=[46, 50, 48, 46, 43, 45, 54, 50, 41, 41], utr_sizes_from_reads=[153, 148, 148, 144, 375, 151, 148, 148, 142, 138])

In [30]:
counter = 0
control = 0
barcodes_real = vivo_anno.CellBarcode
for barcodes in ft.read_barcodes:
    control +=1
    if set(barcodes).issubset(barcodes_real):
        counter += 1
print(counter, control, sep='\n\n')

32

71


In [31]:
%time filts = peak_calling.filter_barcodes(ft, vivo_anno_path)
print('\n\n')
len(filts)

2021-06-30 11:25:24,185 - INFO - Filtering cell barcodes
CPU times: user 184 ms, sys: 3.03 ms, total: 187 ms
Wall time: 187 ms





71

In [32]:
counter = 0
control = 0
barcodes_real = vivo_anno.CellBarcode
for barcodes in filts.read_barcodes:
    control +=1
    if set(barcodes).issubset(barcodes_real):
        counter += 1
print(counter, control, sep='\n\n')

71

71


In [33]:
print(positions_counts_merged.ENSMUSG00000025917, ft.ENSMUSG00000025917, filts.ENSMUSG00000025917, sep='\n\n')

MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024824, 10024829, 10024829, 10024833, 10024602, 10024826, 10024829, 10024829, 10024835, 10024839], read_strands=['+', '+', '+', '+', '+', '+', '+', '+', '+', '+'], read_barcodes=['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2'], read_umis=['TGGCTTATA', 'AACGTAACA', 'ACCTCCGGT', 'ACCCGCACT', 'TCACGTGAG', 'TCCTAAATG', 'ATGATCGGT', 'TCCAGGAAT', 'TTTAGAACT', 'GCTAGGACG'], read_lengths=[46, 50, 48, 46, 43, 45, 54, 50, 41, 41], utr_sizes_from_reads=[153, 148, 148, 144, 375, 151, 148, 148, 142, 138])

MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024824, 10024829, 10024829, 10024833, 10024602, 10024826, 10024829, 10024829, 10024835, 10024839], read_strands=['+', '+', '+', '+', '+', '+', '+', '+', '+', '+'], read_barcod

In [34]:
print(ft.ENSMUSG00000025917['read_barcodes'], filts.ENSMUSG00000025917['read_barcodes'], sep='\n\n')

['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2']

['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2']


In [35]:
%time df, peak_var, obs = peak_calling.generate_peak_counts(filts, annotations=vivo_anno)
print(df)

2021-06-30 11:25:24,519 - INFO - Generating and counting UTR size peaks
Calling peaks for each UTR: 100%|██████████| 71/71 [00:01<00:00, 67.10it/s]
CPU times: user 262 ms, sys: 222 ms, total: 483 ms
Wall time: 1.54 s
                    ENSMUSG00000025917-1  ENSMUSG00000025917-2  \
TCGGGACCAAGTCTGT-1                   0.0                   0.0   
CTTGGCTGTCCGACGT-2                   0.0                   0.0   
GATCAGTTCATGCATG-1                   0.0                   0.0   
AACCATGCAGGAATGC-2                   0.0                   0.0   
GTGCGGTCAGGGCATA-1                   0.0                   0.0   
...                                  ...                   ...   
ACATACGCATCAGTCA-2                   0.0                   0.0   
GTGCATACACCATCCT-1                   0.0                   0.0   
GTCGGGTTCAACACTG-1                   0.0                   0.0   
GGCTGGTGTGATGCCC-1                   0.0                   0.0   
CGGTTAACAATGAATG-2                   0.0                 

In [36]:
peak_var

,utr_size
ENSMUSG00000025917-1,149.0
ENSMUSG00000025917-2,375.0
ENSMUSG00000025925-1,346.5
ENSMUSG00000037509-1,906.0
ENSMUSG00000026356-1,226.0
...,...
ENSMUSG00000066839-2,76.0
ENSMUSG00000034135-1,1068.0
ENSMUSG00000034135-2,2118.0
ENSMUSG00000031429-1,191.0


In [37]:
count_matrix_anndata = peak_calling.generate_count_matrix_anndata(df, peak_var, obs=obs)

2021-06-30 11:25:26,365 - INFO - Generating Count Matrix


In [38]:
print(count_matrix_anndata.var)

                      utr_size
ENSMUSG00000025917-1     149.0
ENSMUSG00000025917-2     375.0
ENSMUSG00000025925-1     346.5
ENSMUSG00000037509-1     906.0
ENSMUSG00000026356-1     226.0
...                        ...
ENSMUSG00000066839-2      76.0
ENSMUSG00000034135-1    1068.0
ENSMUSG00000034135-2    2118.0
ENSMUSG00000031429-1     191.0
ENSMUSG00000031429-2     710.0

[172 rows x 1 columns]


In [39]:
utr_subsetted

,chrom,start,end,name,score,strand,transcript_id,gene_name
utr_name,,,,,,,,
ENSMUSG00000025917,1,10024601,10024978,ENSMUSG00000025917,0.0,-,ENSMUST00000027050,Cops5
ENSMUSG00000025925,1,15843083,15843470,ENSMUSG00000025925,0.0,+,ENSMUST00000027057,Terf1
ENSMUSG00000037509,1,34811830,34812739,ENSMUSG00000037509,0.0,+,ENSMUST00000159747,Arhgef4
ENSMUSG00000026356,1,128412118,128412347,ENSMUSG00000026356,0.0,-,ENSMUST00000064309,Dars
ENSMUSG00000058267,1,160199611,160199812,ENSMUSG00000058267,0.0,+,ENSMUST00000097193,Mrps14
...,...,...,...,...,...,...,...,...
ENSMUSG00000032002,9,7206776,7208205,ENSMUSG00000032002,0.0,+,ENSMUST00000215683,Dcun1d5
ENSMUSG00000066892,9,20637786,20638376,ENSMUSG00000066892,0.0,-,"ENSMUST00000086458,ENSMUST00000148631",Fbxl12
ENSMUSG00000066839,9,22072246,22072323,ENSMUSG00000066839,0.0,-,ENSMUST00000098937,Ecsit


In [40]:
import anndata
peak_counts = anndata.read_h5ad('/home/ubuntu/mytools/utrcalling_/test_utr_results_2.h5ad')

In [41]:
print(f'Counts:\n{peak_counts.X}\n\nObs:\n{peak_counts.obs}\n\nVar:\n{peak_counts.var}')

Counts:
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]

Obs:
                    CellType   CellAge      pTime
index                                            
CGGACGTTCGTTGACA-1  b'qNSC1'    b'old'   0.862199
AAACGGGCACCGGAAA-1  b'qNSC2'    b'old'  20.304435
GGGATGACATATGAGA-2    b'TAP'  b'young'  28.605613
GCAATCAGTTAAGAAC-1  b'aNSC2'    b'old'  18.794153
GACGCGTTCTTGAGGT-1     b'NB'    b'old'  32.353067
...                      ...       ...        ...
CGGTTAACAATGAATG-2     b'NB'  b'young'  31.315385
AACTCAGTCTGATTCT-2  b'qNSC2'  b'young'  10.137653
GATCAGTAGACAAAGG-2    b'TAP'  b'young'  31.345172
ATGGGAGCAAAGTGCG-2  b'aNSC1'  b'young'  21.889695
ACATGGTGTCAACATC-1  b'qNSC1'    b'old'  22.241106

[1014 rows x 3 columns]

Var:
                      utr_size
index                         
ENSMUSG00000025917-1     149.0
ENSMUSG00000025917-2     375.0
ENSMUSG00000025925-1     

In [42]:
peak_var

,utr_size
ENSMUSG00000025917-1,149.0
ENSMUSG00000025917-2,375.0
ENSMUSG00000025925-1,346.5
ENSMUSG00000037509-1,906.0
ENSMUSG00000026356-1,226.0
...,...
ENSMUSG00000066839-2,76.0
ENSMUSG00000034135-1,1068.0
ENSMUSG00000034135-2,2118.0
ENSMUSG00000031429-1,191.0


In [43]:
%%time
new_ref = pd.DataFrame()
for index, utr in utr_subsetted.iterrows():
    peak_var_2 = peak_var[peak_var.index.str.startswith(index)]
    for index_2, peak in peak_var_2.iterrows():
        new_row = peak.append(utr, ignore_index=False)
        new_row.name = index_2
        new_ref = new_ref.append(new_row, ignore_index=False)
print(new_ref)

                     chrom          end gene_name                name  score  \
ENSMUSG00000025917-1     1   10024978.0     Cops5  ENSMUSG00000025917    0.0   
ENSMUSG00000025917-2     1   10024978.0     Cops5  ENSMUSG00000025917    0.0   
ENSMUSG00000025925-1     1   15843470.0     Terf1  ENSMUSG00000025925    0.0   
ENSMUSG00000037509-1     1   34812739.0   Arhgef4  ENSMUSG00000037509    0.0   
ENSMUSG00000026356-1     1  128412347.0      Dars  ENSMUSG00000026356    0.0   
...                    ...          ...       ...                 ...    ...   
ENSMUSG00000066839-2     9   22072323.0     Ecsit  ENSMUSG00000066839    0.0   
ENSMUSG00000034135-1     9   46224194.0      Sik3  ENSMUSG00000034135    0.0   
ENSMUSG00000034135-2     9   46224194.0      Sik3  ENSMUSG00000034135    0.0   
ENSMUSG00000031429-1     X  140949140.0    Psmd10  ENSMUSG00000031429    0.0   
ENSMUSG00000031429-2     X  140949140.0    Psmd10  ENSMUSG00000031429    0.0   

                            start stran

In [51]:
print(positions_counts_merged.utr_ids)

['ENSMUSG00000025917', 'ENSMUSG00000025925', 'ENSMUSG00000037509', 'ENSMUSG00000026356', 'ENSMUSG00000058267', 'ENSMUSG00000004207', 'ENSMUSG00000020231', 'ENSMUSG00000001120', 'ENSMUSG00000071072', 'ENSMUSG00000025358', 'ENSMUSG00000051355', 'ENSMUSG00000001036', 'ENSMUSG00000020840', 'ENSMUSG00000020521', 'ENSMUSG00000020739', 'ENSMUSG00000021147', 'ENSMUSG00000071547', 'ENSMUSG00000033004', 'ENSMUSG00000033055', 'ENSMUSG00000022390', 'ENSMUSG00000022529', 'ENSMUSG00000022684', 'ENSMUSG00000046962', 'ENSMUSG00000057388', 'ENSMUSG00000001525', 'ENSMUSG00000024067', 'ENSMUSG00000002477', 'ENSMUSG00000057766', 'ENSMUSG00000024870', 'ENSMUSG00000040549', 'ENSMUSG00000027254', 'ENSMUSG00000005804', 'ENSMUSG00000037514', 'ENSMUSG00000068134', 'ENSMUSG00000027455', 'ENSMUSG00000027506', 'ENSMUSG00000028057', 'ENSMUSG00000028098', 'ENSMUSG00000028295', 'ENSMUSG00000028483', 'ENSMUSG00000062627', 'ENSMUSG00000028649', 'ENSMUSG00000059447_1', 'ENSMUSG00000029177_1', 'ENSMUSG00000037373', 'ENSM